# Data Cleaning Phases
The followng code is segregated by markdowns each approximately corressponding to a factor we deemed to have a relation to malaria rates. These however,are only data that were accessible via csvs, API requests will come wtih a separate notebook.

In [59]:
import pandas as pd
from scipy import stats

Data sources:

World Health Organization:
https://www.who.int/data/gho/data/indicators/indicator-details/GHO/number-confirmed-malaria-cases

Our World in Data:
https://data.worldbank.org/indicator/NY.GDP.PCAP.CD

# Phase 1: Malaria
Datacleaning WHO malaria data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ZacharyMalonjao/Capstone2-A3101-Group2-SY26-27/refs/heads/main/spreadsheets/malaria.csv')
df.head()

,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,MALARIA_CONF_CASES,Number of confirmed malaria cases,numeric,EUR,Europe,Country,TKM,Turkmenistan,Year,2024,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,EN,2025-12-18T16:00:00.000Z
1,MALARIA_CONF_CASES,Number of confirmed malaria cases,numeric,EUR,Europe,Country,UZB,Uzbekistan,Year,2024,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,EN,2025-12-18T16:00:00.000Z
2,MALARIA_CONF_CASES,Number of confirmed malaria cases,numeric,AMR,Americas,Country,BLZ,Belize,Year,2024,...,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,EN,2025-12-18T16:00:00.000Z
3,MALARIA_CONF_CASES,Number of confirmed malaria cases,numeric,SEAR,South-East Asia,Country,TLS,Timor-Leste,Year,2024,...,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,EN,2025-12-18T16:00:00.000Z
4,MALARIA_CONF_CASES,Number of confirmed malaria cases,numeric,AFR,Africa,Country,BEN,Benin,Year,2024,...,NaN,NaN,NaN,NaN,NaN,1 739 412,NaN,NaN,EN,2025-12-18T16:00:00.000Z


: 

: 

: 

: 

: 

In [ ]:
#Im going to grab the essential and essentially pivot it lengthwise. You'll see as the code progresses.
df_clean =  df[['Location', 'Period', 'FactValueNumeric', 'ParentLocationCode', 'SpatialDimValueCode']]
df_clean.head()

,Location,Period,FactValueNumeric,ParentLocationCode,SpatialDimValueCode
0,Turkmenistan,2024,0,EUR,TKM
1,Uzbekistan,2024,0,EUR,UZB
2,Belize,2024,1,AMR,BLZ
3,Timor-Leste,2024,1,SEAR,TLS
4,Benin,2024,1739412,AFR,BEN


: 

: 

: 

: 

: 

: 

In [ ]:
#check for duplicates
print(df.groupby(['Location', 'Period']).size().max()) 

1


: 

: 

: 

: 

: 

: 

In [ ]:
#Here's where the pivoting starts
df_wide = df_clean.pivot_table(
    index = 'Location',
    columns = 'Period',
    values='FactValueNumeric',
    aggfunc='sum'
).reset_index()

df_wide.columns = ['Country'] + [f'cases_{yr}' for yr in df_wide.columns[1:]]

#merge back the other metadata
meta = df_clean[['Location', 'ParentLocationCode', 'SpatialDimValueCode']].drop_duplicates()
df_wide = df_wide.merge(meta, left_on='Country', right_on='Location').drop(columns='Location')
#just reorder
df_wide = df_wide[['Country', 'ParentLocationCode', 'SpatialDimValueCode'] + [c for c in df_wide.columns if c.startswith('cases_')]]

print(df_wide.head())
#df_wide.to_csv('malaria_wide.csv', index=False)
#If we're going for a time series study, this datat is cool, but we're doing regression so onward me must press on

       Country ParentLocationCode SpatialDimValueCode  cases_2015  cases_2016  \
0  Afghanistan                EMR                 AFG    119859.0    241233.0   
1      Algeria                AFR                 DZA       747.0       432.0   
2       Angola                AFR                 AGO   2769305.0   3794253.0   
3    Argentina                AMR                 ARG        11.0         9.0   
4      Armenia                EUR                 ARM         2.0         2.0   

   cases_2017  cases_2018  cases_2019  cases_2020  cases_2021  cases_2022  \
0    313086.0    248689.0    173860.0    105295.0     86263.0    125620.0   
1       453.0      1242.0      1014.0      2726.0      1164.0      1292.0   
2   3874892.0   5150575.0   7054978.0   7343696.0   8325921.0   7858860.0   
3        16.0        28.0        22.0        13.0        13.0        10.0   
4         2.0         6.0         NaN         3.0         NaN         2.0   

   cases_2023  cases_2024  
0    180045.0    25752

: 

: 

: 

: 

: 

: 

In [ ]:
#check for empty metadata
print(df_clean['ParentLocationCode'].isna().sum())
print(df_clean['SpatialDimValueCode'].isna().sum())

0
0


: 

: 

: 

: 

: 

: 

In [ ]:
#Sir Edmon said we need to check for normality before we aggregate.
# If atleast one country is not normal, we aggregate the cases by median
#We're using Shapio-Wilk Test for this

year_cols = [c for c in df_wide.columns if c.startswith('cases_')]


results = []
for _, row in df_wide.iterrows():
    values = row[year_cols].dropna().values
    
    # Need at least 3 data points to test
    if len(values) < 3:
        results.append({'Country': row['Country'], 'p_value': None, 'normal': None})
        continue
    
    stat, p = stats.shapiro(values)
    results.append({
        'Country': row['Country'],
        'p_value': round(p, 4),
        'normal': p > 0.05  # True = normal, False = skewed
    })

normality_df = pd.DataFrame(results)
print(normality_df)

#Those are a substantial amount of skewed data, so we're using median for aggregation

                                Country  p_value  normal
0                           Afghanistan   0.4867    True
1                               Algeria   0.0210   False
2                                Angola   0.7020    True
3                             Argentina   0.1664    True
4                               Armenia   0.0021   False
..                                  ...      ...     ...
98   Venezuela (Bolivarian Republic of)   0.0303   False
99                             Viet Nam   0.0375   False
100                               Yemen   0.4698    True
101                              Zambia   0.2106    True
102                            Zimbabwe   0.5008    True

[103 rows x 3 columns]


: 

: 

: 

: 

: 

: 

In [ ]:
df_wide['Median_Malaria_Cases'] = df_wide[year_cols].median(axis=1)

df_final = df_wide[['SpatialDimValueCode', 'Country', 'ParentLocationCode', 'Median_Malaria_Cases']]



  SpatialDimValueCode      Country ParentLocationCode  Median_Malaria_Cases
0                 AFG  Afghanistan                EMR              176952.5
1                 DZA      Algeria                AFR                1089.0
2                 AGO       Angola                AFR             7199337.0
3                 ARG    Argentina                AMR                  14.5
4                 ARM      Armenia                EUR                   2.0


: 

: 

: 

: 

: 

: 

In [ ]:
#do some renaming
df_final = df_final.rename(
    columns={
    'SpatialDimValueCode': 'Country_Code',
    'ParentLocationCode': 'Region_Code',   
    'Country': 'Nation' 
    }
)

: 

: 

: 

: 

: 

: 

In [ ]:
print(df_final.head())

  Country_Code       Nation Region_Code  Median_Malaria_Cases
0          AFG  Afghanistan         EMR              176952.5
1          DZA      Algeria         AFR                1089.0
2          AGO       Angola         AFR             7199337.0
3          ARG    Argentina         AMR                  14.5
4          ARM      Armenia         EUR                   2.0


: 

: 

: 

: 

: 

: 

In [ ]:
len(df_final)
#df_final.to_csv('median_malaria_per_nation.csv', index=False)

103

: 

: 

: 

: 

: 

: 

In [ ]:
print(df_final[df_final['Median_Malaria_Cases'] == 0])
print(f"\nCount: {df_final['Median_Malaria_Cases'].eq(0).sum()}")
#I checked the original dataset and they do have recorded malaria cases but they're so low that the median is zero so its valid

   Country_Code        Nation Region_Code  Median_Malaria_Cases
92          TKM  Turkmenistan         EUR                   0.0
96          UZB    Uzbekistan         EUR                   0.0

Count: 2


: 

: 

: 

: 

: 

: 

# Phase 2: GDP
Data cleaning and engineering GDP per nation, then merging it back to the malaria data


In [ ]:
gdp_df = pd.read_csv('https://raw.githubusercontent.com/ZacharyMalonjao/Capstone2-A3101-Group2-SY26-27/refs/heads/main/spreadsheets/GDP_data/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_121663.csv', skiprows=4)
gdp_df.head()


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN,NaN


: 

: 

: 

: 

: 

: 

In [ ]:
df_final['Country_Code'] = df_final['Country_Code'].str.strip().str.upper()
gdp_df['Country Code'] = gdp_df['Country Code'].str.strip().str.upper()

: 

: 

: 

: 

: 

: 

In [ ]:
# Check what the codes actually look like in each dataset
print(df_final['Country_Code'].head(10).tolist())
print(gdp_df['Country Code'].head(10).tolist())
len(gdp_df)


['AFG', 'DZA', 'AGO', 'ARG', 'ARM', 'AZE', 'BGD', 'BLZ', 'BEN', 'BTN']
['ABW', 'AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG']


266

: 

: 

: 

: 

: 

: 

In [ ]:
#find the countries in the gdp dataset that dont have a corresponding country in our main dataset

unmatched = gdp_df[~gdp_df['Country Code'].isin(df_final['Country_Code'])]
print(f"Unmatched countries: {len(unmatched)}")
print(unmatched['Country Name'].tolist())

Unmatched countries: 165
['Aruba', 'Africa Eastern and Southern', 'Africa Western and Central', 'Albania', 'Andorra', 'Arab World', 'United Arab Emirates', 'American Samoa', 'Antigua and Barbuda', 'Australia', 'Austria', 'Belgium', 'Bulgaria', 'Bahrain', 'Bahamas, The', 'Bosnia and Herzegovina', 'Belarus', 'Bermuda', 'Barbados', 'Brunei Darussalam', 'Canada', 'Central Europe and the Baltics', 'Switzerland', 'Channel Islands', 'Chile', 'Caribbean small states', 'Cuba', 'Curacao', 'Cayman Islands', 'Cyprus', 'Czechia', 'Germany', 'Dominica', 'Denmark', 'East Asia & Pacific (excluding high income)', 'Early-demographic dividend', 'East Asia & Pacific', 'Europe & Central Asia (excluding high income)', 'Europe & Central Asia', 'Egypt, Arab Rep.', 'Euro area', 'Spain', 'Estonia', 'European Union', 'Fragile and conflict affected situations', 'Finland', 'Fiji', 'France', 'Faroe Islands', 'Micronesia, Fed. Sts.', 'United Kingdom', 'Gibraltar', 'Greece', 'Grenada', 'Greenland', 'Guam', 'High inco

: 

: 

: 

: 

: 

: 

In [ ]:
missing_gdp = df_final[~df_final['Country_Code'].isin(gdp_df['Country Code'])]
print(f"Malaria countries missing GDP: {len(missing_gdp)}")
print(missing_gdp['Nation'].tolist())
#Both are technically french territories so we can remove them.

Malaria countries missing GDP: 2
['French Guiana', 'Mayotte']


: 

: 

: 

: 

: 

: 

In [ ]:
#Drop french guyana and mayotte
df_final = df_final[df_final['Country_Code'].isin(gdp_df['Country Code'])]
print(len(df_final))  # should be 101

101


: 

: 

: 

: 

: 

: 

In [ ]:
print(gdp_df.columns.tolist())

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


: 

: 

: 

: 

: 

: 

In [ ]:
gdp_df = gdp_df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code','1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', 'Unnamed: 70'])
print(gdp_df.columns.tolist())

KeyError: "['Country Name', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', 'Unnamed: 70'] not found in axis"

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 